In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import scipy.io as sio

data = sio.loadmat("fields_002000.mat", squeeze_me=True)
# data = json.load(open("fields_002000.json", "r"))
slice_index = 0     # the plane of grid index slice_index along the lattice vector a, 0 to nx[0] - 1


def entry(container, key):
    """container[key] of a dict (JSON) or of a MATLAB structure read by loadmat; None if absent."""
    names = container.dtype.names if isinstance(container, np.ndarray) else container
    if names is None or key not in names:
        return None
    value = container[key]
    return value.item() if isinstance(value, np.ndarray) and value.dtype == object and value.ndim == 0 else value


nx = np.asarray(data['nx'], dtype=int)
lx = np.asarray(data['lx'], dtype=float)


def concentration(name):
    """phi of a monomer type: phi_<name>, the real part of CL-FTS, or the phi structure of older files."""
    if 'phi_' + name in data.keys():
        return np.asarray(data['phi_' + name])
    if 'phi_' + name + '_real' in data.keys():
        return np.asarray(data['phi_' + name + '_real'])
    return np.asarray(entry(data['phi'], name))


phi_A = concentration('A').reshape(nx)
if not 0 <= slice_index < nx[0]:
    raise ValueError(f"slice_index must be 0 to {nx[0] - 1}, got {slice_index}")
print(f"phi_A min: {phi_A.min():.4f}, max: {phi_A.max():.4f}")
plane = phi_A[slice_index]      # [j, k]: along b, along c

# The angle alpha between b and c in degrees: SCFT results, the checkpoint settings of fields
# files, or the input of older files
input_angles = entry(data['initial_params'], 'angles') if 'initial_params' in data.keys() else None
if 'angles' in data.keys():
    alpha = float(np.atleast_1d(data['angles'])[0])
elif 'checkpoint_settings_json' in data.keys():
    alpha = float(json.loads(str(data['checkpoint_settings_json']))['angles'][0])
elif input_angles is not None and np.size(input_angles) == 3:
    alpha = float(np.atleast_1d(input_angles)[0])
else:
    alpha = 90.0
    print("The file holds no lattice angles: the plane is drawn rectangular.")
b = lx[1]*np.array([1.0, 0.0])
c = lx[2]*np.array([np.cos(np.deg2rad(alpha)), np.sin(np.deg2rad(alpha))])

# A hexagonal-axes space group (Hall numbers 430-488 except the rhombohedral-axes settings)
# samples b at the nodes
offset_b = 0.5
hall = int(data['space_group_hall_number']) if 'space_group_hall_number' in data.keys() else 0
if 430 <= hall <= 488 and hall not in (434, 437, 445, 451, 453, 459, 461):
    offset_b = 0.0

# Corners of the grid cells: each value fills the cell around its point, (j + offset_b)/n along b
# and (k + 1/2)/n along c, as in the solvers; b is drawn along x and c in the xy plane
fb, fc = np.meshgrid((np.arange(nx[1] + 1) + offset_b - 0.5)/nx[1], np.arange(nx[2] + 1)/nx[2], indexing='ij')
X, Y = fb*b[0] + fc*c[0], fb*b[1] + fc*c[1]

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.pcolormesh(X, Y, plane, cmap=cm.jet, vmin=0.0, vmax=1.0)
ax.set(title=f'phi_A, plane {slice_index} along a', xlabel='x', ylabel='y', aspect='equal')
fig.colorbar(im, ax=ax)
fig.show()

# fig.savefig("2d_slice_image.png", dpi=300, bbox_inches='tight', pad_inches=0)